# 02 · Classical models — TF-IDF + LogReg vs TF-IDF + metadata + XGBoost
This notebook calls the same code as `python -m src.train_classical`, then digs into the results: threshold choice, class weights vs SMOTE, explanations, and generalisation to the custom Indian set.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import logging; logging.basicConfig(level=logging.INFO)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from IPython.display import Image, display
from src.config import MODELS_DIR, MODEL_META_JSON, RESULTS_DIR, RESULTS_CSV, set_seed
from src.train_classical import train_and_compare, prepare_frame
set_seed()

## 1. Train & compare all models
Set `QUICK = True` for a 30 % sample while experimenting (~1 min).

In [ ]:
QUICK = False
table = train_and_compare(quick=QUICK, use_smote=True, readme=False)

In [ ]:
cols = ["model", "mode", "precision", "recall", "f1", "pr_auc", "accuracy", "tp", "fn", "fp", "threshold"]
for ds in ["emscad_test", "custom_india"]:
    print(f"\n=== {ds} ===")
    display(table[table.dataset == ds][cols].round(3))

### Reading the table
* `always_real_baseline` has ~95 % **accuracy** but **0 recall** → accuracy is misleading.
* Compare models by **PR-AUC** (threshold-independent) and **recall** at the tuned threshold.
* `hybrid` = model + rule-based red flags (what the app uses). Watch how much it helps on **custom_india**.

In [ ]:
for ds in ["emscad_test", "custom_india"]:
    p = RESULTS_DIR / f"pr_curve_{ds}.png"
    if p.exists(): display(Image(str(p)))

## 2. Effect of the decision threshold (best model, validation set)

In [ ]:
from src.data_loader import load_emscad, stratified_split
from sklearn.metrics import precision_recall_curve
meta = json.loads(MODEL_META_JSON.read_text())
best = meta["best"]; print("Best model:", best, meta["models"][best])
model = joblib.load(MODELS_DIR / meta["models"][best]["file"])
_, val, _ = stratified_split(load_emscad() if not QUICK else load_emscad().groupby("label", group_keys=False).sample(frac=0.3, random_state=42))
val_f = prepare_frame(val)
p_val = model.predict_proba(val_f)[:, 1]
prec, rec, thr = precision_recall_curve(val_f["label"], p_val)
plt.figure(figsize=(6, 3.5))
plt.plot(thr, prec[:-1], label="precision"); plt.plot(thr, rec[:-1], label="recall")
plt.axvline(meta["models"][best]["threshold"], ls="--", c="k", label="chosen (max F2)")
plt.xlabel("threshold"); plt.legend(); plt.title("Precision / recall vs threshold (validation)"); plt.show()

Lowering the threshold catches more scams (↑ recall) at the cost of more false alarms (↓ precision). We pick the threshold that maximises **F2** (recall weighted 2× precision) **on validation only**, then report test metrics once.

## 3. Class weights vs SMOTE

In [ ]:
t = table[(table.dataset == "emscad_test") & (table["mode"] == "model_only") & (table.model != "always_real_baseline")]
t = t.assign(imbalance=np.where(t.model.str.endswith("_smote"), "SMOTE", "class weights"),
             family=t.model.str.replace("_smote", "", regex=False))
t.pivot_table(index="family", columns="imbalance", values=["recall", "precision", "pr_auc"]).round(3)

SMOTE creates synthetic minority samples by interpolating between neighbours in TF-IDF space. With very sparse, high-dimensional text vectors those interpolations are not real documents, so SMOTE often helps less than simply re-weighting the loss — and it makes training slower. Check what your run shows.

## 4. Explanations for individual posts

In [ ]:
from src.predict import Predictor
from src.explain import segments_to_html
from IPython.display import HTML
pred = Predictor()
post = ("URGENT!!! Work from home data entry. Earn ₹50,000 per week. No interview, direct joining. "
        "Pay registration fee ₹999 and WhatsApp 9000000001 now!")
res = pred.predict(post, email="jobs.hr@gmail.com")
print(res["label"], res["risk_score"], res["explanation_method"])
display(pd.DataFrame(res["highlighted_words"]))
display(pd.DataFrame(res["red_flags"])[["name", "triggered", "evidence"]])
HTML(segments_to_html(res["highlighted_segments"]))

In [ ]:
# LIME (model-agnostic) for the same post — requires `pip install lime`
try:
    res_lime = pred.predict(post, email="jobs.hr@gmail.com", explain="lime")
    display(pd.DataFrame(res_lime["highlighted_words"]))
except Exception as e:
    print("LIME unavailable:", e)

## 5. Errors on the custom Indian set

In [ ]:
from src.data_loader import load_custom_india
india = load_custom_india()
rows = []
for r in india.itertuples():
    o = pred.predict(r.text, r.company, r.email, r.salary, explain="none")
    rows.append({"label": r.label, "pred": int(o["is_fake"]), "risk": o["risk_score"],
                 "model_p": o["model_probability"], "rule": o["rule_score"], "text": r.text[:90]})
res_df = pd.DataFrame(rows)
print("Misclassified:")
res_df[res_df.label != res_df.pred]

**Interpretation:** a model trained only on EMSCAD often assigns low probability to Indian scams (different vocabulary: ₹, WhatsApp, registration fee). The rule layer catches many of them — this is the generalisation gap you should talk about in interviews. The fix in the long run is collecting **real** Indian data and training on it.

In [ ]:
# Write the final table into README.md
from src.evaluate import update_readme
update_readme(pd.read_csv(RESULTS_CSV))